### Get understanding of the correlation between the data

In [ ]:
import pandas as pd
import numpy as numpy
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
df_dtypes_pcd = pd.read_parquet('../data/processed/credit_dtypes_pcd.parquet')
display(df_dtypes_pcd.head(5))

___
## Attribute und Häufigkeiten

In [ ]:


fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))

axes[0].hist(df_dtypes_pcd['person_income'] / 1000, bins=50, range=(0, 300), color='blue')
axes[0].set_title('Abb. 1: Einkommen der Kreditnehmer')
axes[0].set_xlabel('Einkommen in Tausend Euro'); axes[0].set_ylabel('Häufigkeit')

axes[1].hist(df_dtypes_pcd['person_age'], bins=50, range=(18, 60), color='orange')
axes[1].set_title('Abb. 2: Alter der Kreditnehmer')
axes[1].set_xlabel('Alter in Jahren'); axes[1].set_ylabel('Häufigkeit')

axes[2].hist(df_dtypes_pcd['loan_amnt'] / 1000, bins=30, color='red')
axes[2].set_title('Abb. 3: Einkommenshöhe')
axes[2].set_xlabel('Kredithöhe in Tsd.'); axes[2].set_ylabel('Häufigkeit')


plt.tight_layout(); plt.show()

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Abb. 4: loan_intent
counts = df_dtypes_pcd['loan_intent'].value_counts()
axes[0, 0].bar(counts.index, counts.values, color='steelblue')
axes[0, 0].set_title('Abb. 4: Verwendungszweck')
axes[0, 0].set_ylabel('Häufigkeit')
axes[0, 0].tick_params(axis='x', rotation=45)

# Abb. 5: person_home_ownership
counts = df_dtypes_pcd['person_home_ownership'].value_counts()
axes[0, 1].bar(counts.index, counts.values, color='seagreen')
axes[0, 1].set_title('Abb. 5: Wohnsituation')
axes[0, 1].set_ylabel('Häufigkeit')

# Abb. 6: cb_person_default_on_file
counts = df_dtypes_pcd['cb_person_default_on_file'].value_counts(dropna=False)
axes[1, 0].bar(counts.index.astype(str), counts.values, color='darkorange')
axes[1, 0].set_title('Abb. 6: Früherer Kreditausfall')
axes[1, 0].set_ylabel('Häufigkeit')

# Abb. 7: loan_status
counts = df_dtypes_pcd['loan_status'].value_counts()
counts.index = counts.index.map({False: 'zahlt zurück', True: 'Ausfall'})
axes[1, 1].bar(counts.index, counts.values, color='firebrick')
axes[1, 1].set_title('Abb. 7: Kreditstatus')
axes[1, 1].set_ylabel('Häufigkeit')

plt.tight_layout(); plt.show()

---
## Ausfallquoten 

In [ ]:
df_plot = df_dtypes_pcd.copy()
df_plot['altersgruppe'] = pd.cut(df_plot['person_age'], bins=[20, 25, 30, 40, 50, 100],
                                 labels=['20–25', '26–30', '31–40', '41–50', '50+'])

merkmale = {
    'loan_intent': 'Verwendungszweck',
    'person_home_ownership': 'Wohnsituation',
    'altersgruppe': 'Altersgruppe',
    'cb_person_default_on_file': 'Früherer Ausfall',
}

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
gesamtquote = df_plot['loan_status'].mean() * 100

for ax, (spalte, titel) in zip(axes.flat, merkmale.items()):
    q = df_plot.groupby(spalte, observed=True)['loan_status'].mean().mul(100)
    ax.bar(q.index.astype(str), q.values, color='firebrick')
    ax.axhline(gesamtquote, color='grey', linestyle='--', label=f'Gesamt: {gesamtquote:.1f} %')
    ax.set_title(f'Ausfallquote nach {titel}')
    ax.set_ylabel('Ausfallquote in %')
    ax.tick_params(axis='x', rotation=45)
    ax.legend()

plt.tight_layout(); plt.show()

---
## Heatmap
Kaum interessant, man findet nur logische Korrelation untere den restlichen Attributen. 

In [ ]:
import seaborn as sns

df_corr = df_dtypes_pcd.copy()

bool_cols = ['loan_status', 'cb_person_default_on_file']
df_corr[bool_cols] = df_corr[bool_cols].astype(float)

df_corr = pd.get_dummies(df_corr, columns=['person_home_ownership', 'loan_intent'], dtype=float)

corr = df_corr.corr(method='spearman')

plt.figure(figsize=(14, 11))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, center=0)
plt.title('Abb. 9: Korrelationsmatrix aller Merkmale')
plt.tight_layout(); plt.show()

## Gezielte Untersuchung der Korrelation aller Attribute mit dem Zahlungsausfall

Ehrlich finde ich hier nichts ehrlich signifikantes.
dass die im nachhinein hinzugefügte loan_int_rate etwas positiv korreliert war zu erwarten.

person_home_ownership_RENT ist mit r=0.24 knapp über der 0.2 grenze und weist eine ganz leichte korrelation erstmal auf. Interessant ist, dass MORTAGE exakt das gegenteil mit r= -0.19 aussagt. Hier scheint also doch eine wenn auch leichte Korrelation zwischen der Wohnsituation und der loan_status zu existieren.

Zudem bedeutet der Wert von person_income r = -0.27 darauf hin, dass Menschen mit höherem Einkommen leicht dazu neigen ihren Kredit auch abzubezahlen.

In [ ]:
ziel = corr[['loan_status']].drop('loan_status').sort_values('loan_status', ascending=False)

plt.figure(figsize=(8, 4))
sns.heatmap(ziel, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, center=0)
plt.title('Korrelation mit Zahlungsausfall')
plt.tight_layout(); plt.show()

---
## Scatterplot
Der ist recht wenig aussagend, es würde sich mehr anbieten die Attribute vergleichend nebeneinander aufzuführen, jenachdem ob loan_status t/f ist


In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(data=df_dtypes_pcd, x='person_income', y='loan_amnt',
                hue='loan_status', alpha=0.3, s=10, palette={False: 'orange', True: 'blue'})
plt.xlim(0, 300_000)
plt.xlabel('Einkommen'); plt.ylabel('Kredithöhe')
plt.title('Abb. 10: Einkommen vs. Kredithöhe nach Kreditstatus')
plt.show()

---
## Bessere Lösung zum Vergleich der Attribute bzgl. loan_status
Das ist jetzt auch nicht sonderlich aussagekräftig, da sich auch die Zinsen am besten komplett aus dem Datensatz verabschieden sollten.
- Einkommen
- Kredithöhe
- Beschäftigungsdauer
.. sind somit die einzigen Attribute denen man im direkten Vergleich etwas abgewinnen kann

Das ganze Erkennt man ein bisschen in der Heatmap aber auch nur wenn man will XD

Wesentlich aufschlussreicher sind jedoch
- Verwendungszweck des Kredits
- Wohnsituation des Schuldners

Hier sieht man große Unterschiede. 

In [ ]:


cat_cols = {
    'person_home_ownership': 'Wohnsituation',
    'loan_intent': 'Verwendungszweck',
    'cb_person_default_on_file': 'Früherer Ausfall',
}

fig, axes = plt.subplots(3, 3, figsize=(16, 14))
palette = {False: 'steelblue', True: 'firebrick'}
axes = axes.flat

# Numerische Merkmale: Boxplots
for ax, (spalte, titel) in zip(axes, num_cols.items()):
    sns.boxplot(data=df_dtypes_pcd, x='loan_status', y=spalte, hue='loan_status',
                palette=palette, showfliers=False, legend=False, ax=ax)
    ax.set_title(titel, fontsize=13)
    ax.set_xlabel(''); ax.set_ylabel('')
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Zahlt zurück', 'Ausfall'])

    mediane = df_dtypes_pcd.groupby('loan_status')[spalte].median()
    for i, m in enumerate(mediane):
        ax.text(i, m, f'{m:,.0f}' if m > 100 else f'{m:.1f}',
                ha='center', va='bottom', fontweight='bold', color='white',
                bbox=dict(facecolor='black', alpha=0.6, pad=2, edgecolor='none'))

# Kategorische Merkmale: Anteile in % pro Gruppe
for ax, (spalte, titel) in zip(axes, cat_cols.items()):
    anteile = pd.crosstab(df_dtypes_pcd[spalte].astype(str), df_dtypes_pcd['loan_status'],
                          normalize='columns') * 100
    anteile.plot(kind='bar', ax=ax, color=[palette[c] for c in anteile.columns], width=0.75)
    ax.set_title(titel, fontsize=13)
    ax.set_xlabel(''); ax.set_ylabel('Anteil in %')
    ax.tick_params(axis='x', rotation=45)
    ax.legend(['Zahlt zurück', 'Ausfall'])

fig.suptitle('Abb. 11: Merkmale nach Kreditstatus (Boxplots ohne Ausreißer)', fontsize=15)
plt.tight_layout(); plt.show()